<a href="https://colab.research.google.com/github/amnaabbasi1234/Amna-Abbasi/blob/main/spotter_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt

RANDOM_STATE = 42

In [6]:
train_test = pd.read_csv('/content/drive/MyDrive/your_folder_name/train_test.csv')
validation = pd.read_csv('/content/drive/MyDrive/your_folder_name/validation.csv')
val_template = pd.read_csv('/content/drive/MyDrive/your_folder_name/validation_predictions_template.csv')
december = pd.read_csv('/content/drive/MyDrive/your_folder_name/december_chart_inputs.csv')

print(train_test.shape, validation.shape, val_template.shape, december.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/your_folder_name/train_test.csv'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from google.colab import files

uploaded = files.upload()


Saving validation-predictions-template.csv to validation-predictions-template.csv


In [8]:
from google.colab import files

uploaded = files.upload()

Saving december-chart-inputs.csv to december-chart-inputs.csv


In [9]:
from google.colab import files

uploaded = files.upload()

Saving score.py to score.py


In [ ]:
from google.colab import files

uploaded = files.upload()

In [12]:
from google.colab import files

uploaded = files.upload()

Saving readme.md to readme.md


In [13]:
from google.colab import files

uploaded = files.upload()

Saving readme.md to readme (1).md


In [14]:
from google.colab import files

uploaded = files.upload()

Saving requirements.txt to requirements.txt


In [15]:
from google.colab import files

uploaded = files.upload()

Saving freight-rate-ml-assessment.pdf to freight-rate-ml-assessment.pdf


In [16]:
from google.colab import files

uploaded = files.upload()

Saving validation.csv to validation.csv


In [18]:
from google.colab import files

uploaded = files.upload()

Saving train-test.csv to train-test.csv


In [20]:
train_test = pd.read_csv('train-test.csv')
validation = pd.read_csv('validation.csv')
val_template = pd.read_csv('validation-predictions-template.csv')
december = pd.read_csv('december-chart-inputs.csv')

print(train_test.shape, validation.shape, val_template.shape, december.shape)

(48000, 14) (12000, 13) (12000, 2) (31, 7)


In [21]:
train_weight_median = train_test['weight'].abs().median()
train_market_median = train_test['market_index'].median()

def clean(df, has_market=True):
    df = df.copy()
    df['weight'] = df['weight'].abs().fillna(train_weight_median)
    if has_market and 'market_index' in df.columns:
        df['market_index'] = df['market_index'].fillna(train_market_median)
    df['date'] = pd.to_datetime(df['date'])
    return df

train_test = clean(train_test)
validation = clean(validation)
december = clean(december, has_market=False)

print("Remaining nulls (train):", train_test[['weight','market_index']].isnull().sum().sum())

Remaining nulls (train): 0


In [22]:
train_test['rpm'] = train_test['posted_rate'] / train_test['distance']
lane_rpm = train_test.groupby(['pickup', 'delivery'])['rpm'].mean().rename('lane_avg_rpm')
global_avg_rpm = train_test['rpm'].mean()

def add_features(df):
    df = df.copy()
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_year'] = df['date'].dt.dayofyear
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    if 'quote_signal' in df.columns:
        df['distance_x_quote'] = df['distance'] * df['quote_signal']
    df = df.merge(lane_rpm, on=['pickup', 'delivery'], how='left')
    df['lane_avg_rpm'] = df['lane_avg_rpm'].fillna(global_avg_rpm)
    return df

train_test = add_features(train_test)
validation = add_features(validation)
december = add_features(december)

feature_cols = ['distance', 'weight', 'market_index', 'quote_signal',
                 'distance_x_quote', 'lane_avg_rpm', 'equipment',
                 'day_of_week', 'doy_sin', 'doy_cos']

reduced_feature_cols = ['distance', 'weight', 'lane_avg_rpm', 'equipment',
                         'day_of_week', 'doy_sin', 'doy_cos']

target_col = 'posted_rate'

In [23]:
train_test_sorted = train_test.sort_values('date')
cutoff = train_test_sorted['date'].quantile(0.8)
tr = train_test_sorted[train_test_sorted['date'] <= cutoff]
ho = train_test_sorted[train_test_sorted['date'] > cutoff]
print(f"Train: {len(tr)} rows ({tr['date'].min()} to {tr['date'].max()})")
print(f"Holdout: {len(ho)} rows ({ho['date'].min()} to {ho['date'].max()})")

X_tr, y_tr = tr[feature_cols], tr[target_col]
X_ho, y_ho = ho[feature_cols], ho[target_col]
X_tr_reduced, X_ho_reduced = tr[reduced_feature_cols], ho[reduced_feature_cols]

Train: 38477 rows (2025-01-01 00:00:00 to 2025-08-31 00:00:00)
Holdout: 9523 rows (2025-09-01 00:00:00 to 2025-10-31 00:00:00)


In [24]:
categorical = ['equipment']

def make_preprocessor():
    # Each Pipeline needs its OWN ColumnTransformer instance -- sharing one across
    # Pipelines causes fitting one to silently corrupt the others.
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ], remainder='passthrough')

def report(name, y_true, y_pred):
    print(f"{name:>26s} | MAE: {mean_absolute_error(y_true,y_pred):8.2f} | "
          f"MAPE: {mean_absolute_percentage_error(y_true,y_pred)*100:5.2f}% | "
          f"R2: {r2_score(y_true,y_pred):.4f}")

# Primary models (full feature set, for validation.csv)
baseline = Pipeline([('prep', make_preprocessor()), ('model', LinearRegression())])
baseline.fit(X_tr, y_tr)

gbm = Pipeline([('prep', make_preprocessor()),
                ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=300))])
gbm.fit(X_tr, y_tr)

print("Primary model (full feature set) holdout performance:")
report("Linear baseline", y_ho, baseline.predict(X_ho))
report("GradientBoosting", y_ho, gbm.predict(X_ho))

# Secondary models (reduced feature set, for december_chart_inputs.csv)
gbm_reduced = Pipeline([('prep', make_preprocessor()),
                        ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=300))])
gbm_reduced.fit(X_tr_reduced, y_tr)

linear_reduced = Pipeline([('prep', make_preprocessor()), ('model', LinearRegression())])
linear_reduced.fit(X_tr_reduced, y_tr)

print("\nSecondary model (reduced feature set) holdout performance:")
report("GradientBoosting (reduced)", y_ho, gbm_reduced.predict(X_ho_reduced))
report("Linear (reduced)", y_ho, linear_reduced.predict(X_ho_reduced))

Primary model (full feature set) holdout performance:
           Linear baseline | MAE:   180.76 | MAPE: 11.10% | R2: 0.8264
          GradientBoosting | MAE:   153.32 | MAPE:  6.90% | R2: 0.8296

Secondary model (reduced feature set) holdout performance:
GradientBoosting (reduced) | MAE:   152.25 | MAPE:  7.24% | R2: 0.8316
          Linear (reduced) | MAE:   196.32 | MAPE: 11.61% | R2: 0.8233


In [25]:
final_model = gbm
final_model.fit(train_test[feature_cols], train_test[target_col])

final_model_reduced = linear_reduced
final_model_reduced.fit(train_test[reduced_feature_cols], train_test[target_col])

val_preds = final_model.predict(validation[feature_cols])
validation_out = validation[['load_id']].copy()
validation_out['predicted_rate'] = val_preds
validation_out = val_template[['load_id']].merge(validation_out, on='load_id', how='left')

assert validation_out['predicted_rate'].isnull().sum() == 0, "Missing predictions!"
validation_out.to_csv('validation_predictions.csv', index=False)
print("Saved validation_predictions.csv:", validation_out.shape)

Saved validation_predictions.csv: (12000, 2)


In [26]:
dec_preds = final_model_reduced.predict(december[reduced_feature_cols])
december_out = december.drop(columns=['predicted_rate']).copy()
december_out['predicted_rate'] = dec_preds
december_out = december_out[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']]
december_out.to_csv('december_chart_inputs.csv', index=False)
print("Saved december_chart_inputs.csv:", december_out.shape)

Saved december_chart_inputs.csv: (31, 7)


In [27]:
!pip install -r requirements.txt -q
!python score.py --predictions validation_predictions.csv --december-predictions december_chart_inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.


In [28]:
import pandas as pd
vp = pd.read_csv('validation_predictions.csv')
print(vp.shape)  # should be (12000, 2)
print(vp['predicted_rate'].describe())

dec = pd.read_csv('december_chart_inputs.csv')
print(dec[['date','predicted_rate']])

(12000, 2)
count    12000.000000
mean      2338.284999
std       1398.006619
min        288.564833
25%       1248.226684
50%       1988.075220
75%       3303.971900
max       9744.541999
Name: predicted_rate, dtype: float64
          date  predicted_rate
0   2025-12-01      727.465815
1   2025-12-02      723.850892
2   2025-12-03      720.258315
3   2025-12-04      716.688292
4   2025-12-05      713.141021
5   2025-12-06      709.616696
6   2025-12-07      706.115503
7   2025-12-08      722.901687
8   2025-12-09      719.447290
9   2025-12-10      716.016543
10  2025-12-11      712.609604
11  2025-12-12      709.226624
12  2025-12-13      705.867749
13  2025-12-14      702.533116
14  2025-12-15      719.486919
15  2025-12-16      716.201152
16  2025-12-17      712.939995
17  2025-12-18      709.703557
18  2025-12-19      706.491938
19  2025-12-20      703.305233
20  2025-12-21      700.143527
21  2025-12-22      717.270966
22  2025-12-23      714.159490
23  2025-12-24      711.073228
2